In [1]:
import pandas as pd
import numpy as np
from sklearn.pipeline import Pipeline
from sklearn.cluster import KMeans, SpectralClustering, DBSCAN, OPTICS
from sklearn.metrics import silhouette_score, davies_bouldin_score
from sklearn.preprocessing import StandardScaler, MinMaxScaler

In [2]:
def pipeline_cluster(model, df):
    ## En caso de necesitar modificar el preprocesamiento, hacerlo aquí
    preprocessor = Pipeline(steps=[
        ('scaler', MinMaxScaler())
    ])

    ## Pipeline final
    pipeline = Pipeline(steps=[
        ('preprocessing', preprocessor),
        ('model', model)
    ])

    return pipeline 

In [3]:
def eval(pipeline, X):

    ## Ajustar el pipeline
    pipeline.fit(X)

    ## Obtener las etiquetas predichas
    if hasattr(pipeline.named_steps['model'], 'labels_'):
        labels = pipeline.named_steps['model'].labels_
    else:
        labels = pipeline.named_steps['model'].predict(X)

    ## Calcular métricas de evaluación
    silhouette = silhouette_score(X, labels)
    davies_bouldin = davies_bouldin_score(X, labels)

    return silhouette, davies_bouldin, labels

### Ejemplo

In [4]:
features = pd.read_csv('features_output_1.csv.gz', compression='gzip')
##remove interpolated_bandwidth column
features = features.drop(columns=['interpolated_bandwidth'])
labels = pd.read_csv('data/40mhz/original_labels/brc-2002_086400-01-output_true_labels.csv.gz', compression='gzip')

## combine features and labels into a single dataframe without key
data = pd.concat([features, labels], axis=1)

In [ ]:
kmeans_model = KMeans(n_clusters=6, random_state=42)

pipeline = pipeline_cluster(kmeans_model, data.iloc[:,:-3])  ## Exclude last 3 columns which are not features
silhouette, davies_bouldin, cluster_labels = eval(pipeline, data.iloc[:,:-3])  ## Exclude last 3 columns which are not features
print(f'Silhouette Score: {silhouette}')
print(f'Davies-Bouldin Index: {davies_bouldin}')
## add cluster labels to data
data['cluster'] = cluster_labels

## show cluster composition with respect to true labels
print(data.groupby('cluster').apply(lambda x: x.iloc[:,-2:].value_counts()))

In [ ]:
optics_model = OPTICS(min_samples=10, xi=0.05, min_cluster_size=0.1)
pipeline_optics = pipeline_cluster(optics_model, data.iloc[:,:-3])  ## Exclude last 3 columns which are not features
silhouette_optics, davies_bouldin_optics, cluster_labels_optics = eval(pipeline_optics, data.iloc[:,:-3])  ## Exclude last 3 columns which are not features
print(f'OPTICS Silhouette Score: {silhouette_optics}')
print(f'OPTICS Davies-Bouldin Index: {davies_bouldin_optics}')
## add cluster labels to data
data['cluster_optics'] = cluster_labels_optics
## show cluster composition with respect to true labels
print(data.groupby('cluster_optics').apply(lambda x: x.iloc[:,-2:].value_counts()))